# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we print the available record sets (`@id`), list the fields for each record set, and display their `@id`s. These `@id` values are used to precisely reference elements within the dataset.

In [ ]:
# List available record sets and their fields
# Record sets in mlcroissant are available as dataset.record_sets

record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for record_set in record_sets:
        print(f"Record Set: {record_set['@id']} - {record_set.get('name','(no name)')}")
        if 'fields' in record_set and record_set['fields']:
            for field in record_set['fields']:
                print(f"  Field: {field['@id']} ({field.get('name','')}) - type: {field.get('dataType','')}")
        else:
            print("  [No fields in this record set]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** If you didn't see any record sets in the previous cell, this indicates the dataset metadata does not enumerate them directly. For the FAIR² Mass Spectrometry Analysis dataset, we will obtain the available record sets via the dataset object's `record_sets` property, which automatically parses the Croissant schema.

In [ ]:
import warnings

record_sets = list(dataset.metadata.record_sets)
dataframes = {}
if not record_sets:
    print("No explicit record sets were found. Attempting to list available record set IDs from the dataset interface.")
    all_record_set_ids = dataset.record_sets
    print(f"Found the following record sets: {all_record_set_ids}")
else:
    all_record_set_ids = [r['@id'] for r in record_sets]
    print(f"Found the following record sets: {all_record_set_ids}")

# For this dataset, let's use the first available record set for demonstration 
if len(all_record_set_ids) > 0:
    selected_record_set_id = all_record_set_ids[0]
    print(f"\nExtracting records for record set: {selected_record_set_id}\n")
    try:
        records = list(dataset.records(record_set=selected_record_set_id))
        df = pd.DataFrame(records)
        dataframes[selected_record_set_id] = df
        print(f"Fields/Columns in record set '{selected_record_set_id}':\n{df.columns.tolist()}")
        display(df.head())
    except Exception as ex:
        warnings.warn(f"Could not extract records for record set '{selected_record_set_id}': {ex}")
else:
    print("No record set available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Below, select one numeric field for analysis based on the columns available in your DataFrame.
import numpy as np
import matplotlib.pyplot as plt

# Use the DataFrame loaded above
df = dataframes.get(selected_record_set_id)
if df is not None:
    # Attempt to select a common numeric field for this domain
    possible_numeric_fields = [
        'coverage', 'percent_coverage', 'sequence_coverage', 'peptide_count',
        'MW', 'molecular_weight', 'calculated_pI', 'abundance', 'normalized_abundance'
    ]

    numeric_field = None
    for f in possible_numeric_fields:
        if f in df.columns:
            numeric_field = f
            break

    if numeric_field is not None:
        # Drop NaNs for computation
        filtered_df = df[df[numeric_field].notnull()]

        # Filter data to only include values above a threshold
        threshold = np.nanmedian(filtered_df[numeric_field])
        print(f"Using numeric field '{numeric_field}'. Median value is {threshold}.")
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (showing up to 5 rows):")
        display(filtered_df.head())

        # Normalize field
        normalized = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        filtered_df[numeric_field + '_normalized'] = normalized
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Attempt to group by a likely categorical field
        possible_group_fields = ['description', 'protein_id', 'gene_name', 'modification_type']
        group_field = None
        for g in possible_group_fields:
            if g in filtered_df.columns:
                group_field = g
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped mean {numeric_field} by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No appropriate numeric field found in columns:", df.columns.tolist())
else:
    print("No data available for DataFrame EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. The charts below are examples based on the numeric field selected in the EDA above.

In [ ]:
# Visualization: Histogram and Boxplot of the chosen numeric field
if df is not None and numeric_field:
    plt.figure(figsize=(10, 4))
    plt.hist(df[numeric_field].dropna(), bins=30, alpha=0.7, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    plt.figure(figsize=(2, 6))
    plt.boxplot(df[numeric_field].dropna(), vert=True)
    plt.title(f"Boxplot of {numeric_field}")
    plt.ylabel(numeric_field)
    plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains protein mass spectrometry measurements from extracellular vesicles isolated from stimulated human mast cells.
- Using `mlcroissant`, we loaded the Croissant schema, extracted available record sets by their `@id`, and operated on the record set data.
- Basic EDA, such as record filtering, normalization, and visualization, can be performed directly from Croissant definitions, referencing precise field `@id`s for reproducibility.
- For further downstream analysis, consider exporting the DataFrame or applying domain-specific transformations relevant to proteomics.